In [1]:
import pandas as pd
import os

In [2]:
print(os.getcwd())

c:\Users\Shyamsekar\OneDrive\Desktop\Cart2Insights\Project_Data\notebooks


In [4]:
customers = pd.read_csv(
    "../Data/cleaned/customers_clean.csv"
)

orders = pd.read_csv(
    "../Data/cleaned/orders_clean.csv"
)

order_items = pd.read_csv(
    "../Data/cleaned/order_items_clean.csv"
)

payments = pd.read_csv(
    "../Data/cleaned/payments_clean.csv"
)

reviews = pd.read_csv(
    "../Data/cleaned/reviews_clean.csv"
)

products = pd.read_csv(
    "../Data/cleaned/products_clean.csv"
)

sellers = pd.read_csv(
    "../Data/cleaned/sellers_clean.csv"
)

geolocation = pd.read_csv(
    "../Data/cleaned/geolocation_clean.csv"
)

category_translation = pd.read_csv(
    "../Data/cleaned/category_translation_clean.csv"
)

print("All cleaned datasets loaded successfully!")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Geolocation:", geolocation.shape)
print("Category Translation:", category_translation.shape)

All cleaned datasets loaded successfully!
Customers: (99441, 5)
Orders: (99441, 8)
Order Items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)
Sellers: (3095, 4)
Geolocation: (738327, 5)
Category Translation: (71, 2)


Feature 1: Convert order dates

In [5]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

print(orders[date_columns].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


Feature 2: Purchase year

In [8]:
orders["purchase_year"] = (
    orders["order_purchase_timestamp"].dt.year
)
orders[
    ["order_purchase_timestamp", "purchase_year"]
].head()

,order_purchase_timestamp,purchase_year
0,2017-10-02 10:56:33,2017
1,2018-07-24 20:41:37,2018
2,2018-08-08 08:38:49,2018
3,2017-11-18 19:28:06,2017
4,2018-02-13 21:18:39,2018


Feature 3: Purchase month

In [9]:
orders["purchase_month"] = (
    orders["order_purchase_timestamp"].dt.month
)

Feature 4: Purchase month name

In [11]:
orders["purchase_month_name"] = (
    orders["order_purchase_timestamp"].dt.month_name()
)

orders[
    [
        "order_purchase_timestamp",
        "purchase_year",
        "purchase_month",
        "purchase_month_name"
    ]
].head()

,order_purchase_timestamp,purchase_year,purchase_month,purchase_month_name
0,2017-10-02 10:56:33,2017,10,October
1,2018-07-24 20:41:37,2018,7,July
2,2018-08-08 08:38:49,2018,8,August
3,2017-11-18 19:28:06,2017,11,November
4,2018-02-13 21:18:39,2018,2,February


Feature 5: Purchase day of week

In [12]:
orders["purchase_day_of_week"] = (
    orders["order_purchase_timestamp"].dt.day_name()
)

Feature 6: Delivery days

In [15]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders[
    [
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "delivery_days"
    ]
].head()

orders["delivery_days"] = orders["delivery_days"].round(2)

Feature 7: Estimated delivery days

In [50]:
orders["estimated_delivery_days"] = (
    orders["order_estimated_delivery_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

orders["estimated_delivery_days"] = (
    orders["estimated_delivery_days"].round(2)
)

Feature 8: Delay days

In [21]:
orders["delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

orders["delay_days"] = orders["delay_days"].round(2)

yes/no

In [23]:
orders["is_late"] = (
    orders["order_delivered_customer_date"]
    > orders["order_estimated_delivery_date"]
)

print(orders["is_late"].value_counts(dropna=False))

is_late
False    91614
True      7827
Name: count, dtype: int64


In [25]:
orders["is_late"] = None

delivered_mask = (
    orders["order_delivered_customer_date"].notna()
    & orders["order_estimated_delivery_date"].notna()
)

orders.loc[delivered_mask, "is_late"] = (
    orders.loc[delivered_mask, "order_delivered_customer_date"]
    >
    orders.loc[delivered_mask, "order_estimated_delivery_date"]
)

orders["is_late"].value_counts(dropna=False)

is_late
False    88649
True      7827
None      2965
Name: count, dtype: int64

Feature 10: Order purchase hour

In [26]:
orders["purchase_hour"] = (
    orders["order_purchase_timestamp"].dt.hour
)

Feature 11: Total item value

In [28]:
order_items["total_item_value"] = (
    order_items["price"]
    + order_items["freight_value"]
)

order_items[
    ["price", "freight_value", "total_item_value"]
].head()

,price,freight_value,total_item_value
0,58.90,13.29,72.19
1,239.90,19.93,259.83
2,199.00,17.87,216.87
3,12.99,12.79,25.78
4,199.90,18.14,218.04


Feature 12: Total order value

In [32]:
order_value = (
    order_items
    .groupby("order_id", as_index=False)
    ["total_item_value"]
    .sum()
)

order_value = order_value.rename(
    columns={
        "total_item_value": "total_order_value"
    }
)

order_value.head()

orders = orders.merge(
    order_value,
    on="order_id",
    how="left"
)



In [33]:
orders[
    ["order_id", "total_order_value"]
].head()

,order_id,total_order_value
0,e481f51cbdc54678b7cc49136f2d6af7,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,28.62


Feature 13: Number of items per order

In [36]:
items_per_order = (
    order_items
    .groupby("order_id")
    .size()
    .reset_index(name="total_items")
)

items_per_order.head()

,order_id,total_items
0,00010242fe8c5a6d1ba2dd792cb16214,1
1,00018f77f2f0320c557190d7a144bdd3,1
2,000229ec398224ef6ca0657da4fc703e,1
3,00024acbcdf0a6daa1e931b038114c75,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1


In [39]:
orders = orders.merge(
    items_per_order,
    on="order_id",
    how="left"
)



In [40]:
orders[
    ["order_id", "total_items", "total_order_value"]
].head()

,order_id,total_items,total_order_value
0,e481f51cbdc54678b7cc49136f2d6af7,1.0,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,1.0,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,1.0,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,1.0,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,1.0,28.62


Feature 14: Customer satisfaction category

In [43]:
reviews["review_score"].value_counts().sort_index()

reviews["satisfaction_category"] = reviews["review_score"].map({
    1: "Very Poor",
    2: "Poor",
    3: "Average",
    4: "Good",
    5: "Excellent"
})

reviews[
    ["review_score", "satisfaction_category"]
].head(10)

,review_score,satisfaction_category
0,4,Good
1,5,Excellent
2,5,Excellent
3,5,Excellent
4,5,Excellent
5,1,Very Poor
6,5,Excellent
7,5,Excellent
8,5,Excellent
9,4,Good


In [46]:
print("ORDER FEATURES")
print(
    orders[
        [
            "order_id",
            "purchase_year",
            "purchase_month",
            "purchase_month_name",
            "purchase_day_of_week",
            "purchase_hour",
            "delivery_days",
            "estimated_delivery_days",
            "delay_days",
            "is_late",
            "total_order_value",
            "total_items"
        ]
    ].head()
)

print(orders.columns.tolist())

ORDER FEATURES
                           order_id  purchase_year  purchase_month  \
0  e481f51cbdc54678b7cc49136f2d6af7           2017              10   
1  53cdb2fc8bc7dce0b6741e2150273451           2018               7   
2  47770eb9100c2d0c44946d9cf07ec65d           2018               8   
3  949d5b44dbf5de918fe9c16f97b45f8a           2017              11   
4  ad21c59c0840e6cb83a9ceb5573f8159           2018               2   

  purchase_month_name purchase_day_of_week  purchase_hour  delivery_days  \
0             October               Monday             10           8.44   
1                July              Tuesday             20          13.78   
2              August            Wednesday              8           9.39   
3            November             Saturday             19          13.21   
4            February              Tuesday             21           2.87   

   estimated_delivery_days  delay_days is_late  total_order_value  total_items  
0                    15.54

In [47]:
orders.to_csv(
    "../Data/cleaned/orders_featured.csv",
    index=False
)

order_items.to_csv(
    "../Data/cleaned/order_items_featured.csv",
    index=False
)

reviews.to_csv(
    "../Data/cleaned/reviews_featured.csv",
    index=False
)

print("Feature-engineered files saved successfully!")

Feature-engineered files saved successfully!


In [48]:
print(
    os.path.exists(
        "../Data/cleaned/orders_featured.csv"
    )
)

print(
    os.path.exists(
        "../Data/cleaned/order_items_featured.csv"
    )
)

print(
    os.path.exists(
        "../Data/cleaned/reviews_featured.csv"
    )
)

True
True
True
